
# Publication figures and tables: Planck HFI differential polarization calibration

This notebook converts the current analysis outputs into publication-ready figures and machine-readable/LaTeX tables for an ApJ-style manuscript.

## Scientific scope

The primary result is the consistency of two reconstructions of the **differential polarization-angle calibration**:

\[
\Delta\alpha_{ij}^{\rm MK}=\alpha_i^{\rm MK}-\alpha_j^{\rm MK},
\qquad
\widehat{\Delta\alpha}_{ij}^{\rm RelCal}.
\]

The anchored birefringence estimates are presented as a conditional application:

> Under adoption of the Minami–Komatsu common calibration mode as an anchor, the total-rotation estimator gives the reported values of \(\beta\).

This notebook does **not** interpret the anchor as an independent absolute calibration.

## Important status note

The previously generated `mk_relcal_compare_30mask.ipynb` mixed the 30% RelCal mask with a mask-0 MK likelihood. Its closure outputs are therefore excluded. Only verified mask-0 pair results and the anchored outputs are embedded below. Replace the embedded tables after rerunning the corrected pipeline.


In [ ]:

from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from numpy.linalg import inv, pinv

OUTPUT = Path("paper_outputs")
FIGURES = OUTPUT / "figures"
TABLES = OUTPUT / "tables"
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.9,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def save_figure(fig, stem):
    fig.savefig(FIGURES / f"{stem}.pdf")
    fig.savefig(FIGURES / f"{stem}.png")
    print(f"Saved {stem}.pdf and {stem}.png")


## 1. Verified mask-0 pair-level results

In [ ]:

pair_rows = [["100A-143A", -0.3423, 0.089, -0.3066, 0.1199, 0.1494, 0.0812, -0.24, -0.44], ["100A-217A", -0.2559, 0.0859, -0.3552, 0.1307, 0.1564, 0.0979, 0.63, 1.01], ["100A-353A", -0.1295, 0.0868, -0.088, 0.1688, 0.1898, 0.1452, -0.22, -0.29], ["100A-100B", 0.1052, 0.1022, 0.1051, 0.1487, 0.1804, 0.1087, 0.0, 0.0], ["100A-143B", -0.4623, 0.0881, -0.4522, 0.1168, 0.1463, 0.0776, -0.07, -0.13], ["100A-217B", -0.236, 0.0869, -0.2327, 0.1303, 0.1566, 0.0982, -0.02, -0.03], ["100A-353B", -0.0922, 0.0876, -0.2231, 0.1741, 0.1949, 0.1509, 0.67, 0.87], ["143A-217A", 0.0864, 0.0529, 0.1359, 0.065, 0.0838, 0.0361, -0.59, -1.37], ["143A-353A", 0.2128, 0.0524, 0.2019, 0.0704, 0.0878, 0.0465, 0.12, 0.23], ["143A-100B", 0.4474, 0.0819, 0.4835, 0.1088, 0.1362, 0.0723, -0.27, -0.5], ["143A-143B", -0.12, 0.0594, -0.1002, 0.0712, 0.0927, 0.0378, -0.21, -0.52], ["143A-217B", 0.1062, 0.0531, 0.0369, 0.065, 0.0839, 0.0364, 0.83, 1.9], ["143A-353B", 0.25, 0.0542, 0.236, 0.0734, 0.0912, 0.05, 0.15, 0.28], ["217A-353A", 0.1264, 0.0302, 0.1366, 0.0319, 0.0439, 0.0114, -0.23, -0.9], ["217A-100B", 0.361, 0.0792, 0.3885, 0.1195, 0.1434, 0.0897, -0.19, -0.31], ["217A-143B", -0.2064, 0.0518, -0.2587, 0.0635, 0.0819, 0.0348, 0.64, 1.5], ["217A-217B", 0.0199, 0.0355, 0.0618, 0.0383, 0.0522, 0.0139, -0.8, -3.01], ["217A-353B", 0.1636, 0.0325, 0.1653, 0.0348, 0.0476, 0.0137, -0.04, -0.12], ["353A-100B", 0.2347, 0.0801, 0.3727, 0.1555, 0.175, 0.1342, -0.79, -1.03], ["353A-143B", -0.3328, 0.0513, -0.4219, 0.0687, 0.0857, 0.0453, 1.04, 1.97], ["353A-217B", -0.1065, 0.03, -0.1074, 0.0318, 0.0438, 0.0111, 0.02, 0.08], ["353A-353B", 0.0372, 0.0232, 0.0355, 0.0233, 0.0329, 0.0045, 0.05, 0.38], ["100B-143B", -0.5674, 0.0812, -0.5627, 0.1063, 0.1338, 0.0695, -0.04, -0.07], ["100B-217B", -0.3412, 0.0802, -0.3574, 0.1185, 0.1431, 0.0893, 0.11, 0.18], ["100B-353B", -0.1974, 0.0809, -0.2359, 0.1613, 0.1804, 0.1403, 0.21, 0.27], ["143B-217B", 0.2263, 0.0522, 0.3086, 0.0636, 0.0823, 0.0354, -1.0, -2.32], ["143B-353B", 0.37, 0.0527, 0.4456, 0.0717, 0.089, 0.0484, -0.85, -1.56], ["217B-353B", 0.1438, 0.0323, 0.1432, 0.0348, 0.0475, 0.0131, 0.01, 0.04]]
pairs = pd.DataFrame(pair_rows, columns=[
    "pair", "mk_delta_deg", "mk_sigma_deg",
    "relcal_delta_deg", "relcal_sigma_deg",
    "sigma_independent_deg", "sigma_shared_data_deg",
    "pull_independent", "pull_shared_data"
])
pairs["difference_deg"] = pairs["relcal_delta_deg"] - pairs["mk_delta_deg"]
pairs.head()


In [ ]:

# Publication table: pair-level comparison
pair_table = pairs.rename(columns={
    "pair": "Map pair",
    "mk_delta_deg": r"$\Delta\alpha_{\rm MK}$ [deg]",
    "mk_sigma_deg": r"$\sigma_{\rm MK}$ [deg]",
    "relcal_delta_deg": r"$\Delta\alpha_{\rm RelCal}$ [deg]",
    "relcal_sigma_deg": r"$\sigma_{\rm RelCal}$ [deg]",
    "difference_deg": r"RelCal$-$MK [deg]",
    "pull_shared_data": "Shared-data pull",
})
pair_table.to_csv(TABLES / "table_pairwise_calibration.csv", index=False)
pair_table.to_latex(
    TABLES / "table_pairwise_calibration.tex",
    index=False, escape=False, float_format="%.4f",
    caption="Pairwise differential polarization-angle estimates for the nearly full-sky mask.",
    label="tab:pairwise"
)
pair_table


In [ ]:

# Figure 1: estimator-to-estimator comparison
x = pairs["mk_delta_deg"].to_numpy()
y = pairs["relcal_delta_deg"].to_numpy()
xe = pairs["mk_sigma_deg"].to_numpy()
ye = pairs["relcal_sigma_deg"].to_numpy()

fig, ax = plt.subplots(figsize=(4.8, 4.5))
ax.errorbar(
    x, y, xerr=xe, yerr=ye, fmt="o", ms=4, capsize=2,
    label="Planck HFI map-pair estimates"
)
limit = 1.12 * np.max(np.abs(np.r_[x, y]))
ax.plot([-limit, limit], [-limit, limit], "--", lw=1.0,
        label="Equality of the two estimators")
ax.set(
    xlabel=r"Minami--Komatsu differential angle $\Delta\alpha_{\rm MK}$ [deg]",
    ylabel=r"Birefringence-insensitive differential angle $\Delta\alpha_{\rm RelCal}$ [deg]",
    xlim=(-limit, limit), ylim=(-limit, limit)
)
correlation = np.corrcoef(x, y)[0, 1]
ax.text(0.04, 0.96, rf"Pearson correlation: ${correlation:.3f}$",
        transform=ax.transAxes, va="top")
ax.legend(loc="lower right", frameon=False)
fig.tight_layout()
save_figure(fig, "figure_estimator_consistency")
plt.show()


In [ ]:

# Figure 2: pair-by-pair comparison
order = np.argsort(pairs["mk_delta_deg"].to_numpy())
plot_data = pairs.iloc[order].reset_index(drop=True)
positions = np.arange(len(plot_data))

fig, ax = plt.subplots(figsize=(7.2, 8.2))
ax.errorbar(
    plot_data["mk_delta_deg"], positions - 0.14,
    xerr=plot_data["mk_sigma_deg"], fmt="o", ms=3.5, capsize=2,
    label="Minami--Komatsu calibration differences"
)
ax.errorbar(
    plot_data["relcal_delta_deg"], positions + 0.14,
    xerr=plot_data["relcal_sigma_deg"], fmt="s", ms=3.5, capsize=2,
    label="Birefringence-insensitive relative calibration"
)
ax.axvline(0, lw=0.8)
ax.set_yticks(positions)
ax.set_yticklabels(plot_data["pair"])
ax.set_xlabel(r"Differential polarization angle $\Delta\alpha$ [deg]")
ax.set_ylabel("Planck HFI map pair")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
save_figure(fig, "figure_pairwise_differential_angles")
plt.show()


In [ ]:

# Figure 3: shared-data pulls
plot_data = pairs.sort_values("pull_shared_data").reset_index(drop=True)
positions = np.arange(len(plot_data))

fig, ax = plt.subplots(figsize=(7.2, 8.2))
ax.scatter(plot_data["pull_shared_data"], positions, s=22)
ax.axvline(0, lw=0.8)
ax.axvline(-2, ls="--", lw=0.8)
ax.axvline(2, ls="--", lw=0.8)
ax.axvline(-3, ls=":", lw=0.8)
ax.axvline(3, ls=":", lw=0.8)
ax.set_yticks(positions)
ax.set_yticklabels(plot_data["pair"])
ax.set_xlabel(r"Difference divided by the shared-data uncertainty")
ax.set_ylabel("Planck HFI map pair")
fig.tight_layout()
save_figure(fig, "figure_pairwise_consistency_pulls")
plt.show()



## 2. Exact-gauge network reconstruction

The relative estimator measures only angle differences. We therefore solve the weighted network in a reduced basis, fixing the 143A map to zero exactly. This avoids a soft gauge penalty and yields a reproducible map-level visualization.

The absolute vertical offset in this section has no physical meaning.


In [ ]:

labels = ["100A", "143A", "217A", "353A", "100B", "143B", "217B", "353B"]
reference = "143A"
free_labels = [name for name in labels if name != reference]
index = {name: i for i, name in enumerate(free_labels)}

A = np.zeros((len(pairs), len(free_labels)))
d = pairs["relcal_delta_deg"].to_numpy()
s = pairs["relcal_sigma_deg"].to_numpy()

for row, pair in enumerate(pairs["pair"]):
    left, right = pair.split("-")
    if left != reference:
        A[row, index[left]] = 1.0
    if right != reference:
        A[row, index[right]] = -1.0

W = np.diag(1.0 / s**2)
cov_free = inv(A.T @ W @ A)
alpha_free = cov_free @ A.T @ W @ d

alpha_rel = {reference: 0.0}
sigma_rel = {reference: 0.0}
for name in free_labels:
    alpha_rel[name] = alpha_free[index[name]]
    sigma_rel[name] = np.sqrt(cov_free[index[name], index[name]])

# MK map values from the mask-0 posterior output, centered to the same 143A gauge.
mk_alpha = {
    "100A": -0.33, "143A": 0.02, "217A": -0.07, "353A": -0.20,
    "100B": -0.43, "143B": 0.14, "217B": -0.09, "353B": -0.23,
}
mk_sigma = {
    "100A": 0.14, "143A": 0.12, "217A": 0.11, "353A": 0.11,
    "100B": 0.13, "143B": 0.12, "217B": 0.11, "353B": 0.11,
}
mk_center = mk_alpha[reference]
mk_relative = {name: mk_alpha[name] - mk_center for name in labels}

network = pd.DataFrame({
    "Map": labels,
    "Relative calibration estimator [deg]": [alpha_rel[n] for n in labels],
    "Relative calibration uncertainty [deg]": [sigma_rel[n] for n in labels],
    "MK posterior, common mode removed [deg]": [mk_relative[n] for n in labels],
    "MK marginal uncertainty [deg]": [mk_sigma[n] for n in labels],
})
network.to_csv(TABLES / "table_network_solution.csv", index=False)
network.to_latex(
    TABLES / "table_network_solution.tex", index=False,
    float_format="%.4f",
    caption="Map-level differential calibration network in the exact gauge $\\alpha_{143A}=0$.",
    label="tab:network"
)
network


In [ ]:

# Figure 4: map-level differential calibration network
positions = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(6.3, 4.7))
ax.errorbar(
    [mk_relative[n] for n in labels], positions - 0.12,
    xerr=[mk_sigma[n] for n in labels], fmt="o", capsize=3,
    label="Minami--Komatsu posterior, common mode removed"
)
ax.errorbar(
    [alpha_rel[n] for n in labels], positions + 0.12,
    xerr=[sigma_rel[n] for n in labels], fmt="s", capsize=3,
    label="Birefringence-insensitive network solution"
)
ax.axvline(0, lw=0.8)
ax.set_yticks(positions)
ax.set_yticklabels(labels)
ax.set_xlabel(r"Relative polarization calibration with $\alpha_{143A}=0$ [deg]")
ax.set_ylabel("Planck HFI map")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "figure_network_reconstruction")
plt.show()


## 3. Anchored total-rotation application

In [ ]:

anchored = pd.DataFrame([["MK full network, nearly full sky", 0.3723, 0.1166], ["Anchored estimator, nearly full sky", 0.3657, 0.1153], ["Anchored estimator, 30% Galactic mask", 0.4064, 0.1145]], columns=["Analysis", "beta_deg", "sigma_deg"])
anchored.to_csv(TABLES / "table_anchored_beta.csv", index=False)
anchored.to_latex(
    TABLES / "table_anchored_beta.tex", index=False,
    float_format="%.4f",
    caption="Conditional birefringence estimates obtained after adopting the MK common calibration mode as an anchor.",
    label="tab:anchored"
)
anchored


In [ ]:

# Figure 5: anchored beta summary
positions = np.arange(len(anchored))
fig, ax = plt.subplots(figsize=(6.2, 3.3))
ax.errorbar(
    anchored["beta_deg"], positions,
    xerr=anchored["sigma_deg"], fmt="o", capsize=4
)
ax.axvline(0, lw=0.8)
ax.set_yticks(positions)
ax.set_yticklabels(anchored["Analysis"])
ax.set_xlabel(r"Isotropic rotation angle $\beta$ [deg]")
ax.invert_yaxis()
fig.tight_layout()
save_figure(fig, "figure_anchored_beta_summary")
plt.show()


In [ ]:

per_map = pd.DataFrame([["Nearly full sky", "100A", 0.0789, 0.0825, 0.4054, 0.1199], ["Nearly full sky", "143A", 0.3873, 0.057, 0.3715, 0.1183], ["Nearly full sky", "100B", -0.0744, 0.0739, 0.3572, 0.1335], ["Nearly full sky", "143B", 0.4745, 0.0559, 0.3387, 0.1225], ["30% Galactic mask", "100A", 0.1267, 0.1086, 0.4531, 0.1262], ["30% Galactic mask", "143A", 0.4664, 0.0786, 0.4506, 0.1203], ["30% Galactic mask", "100B", -0.0118, 0.0964, 0.4198, 0.1234], ["30% Galactic mask", "143B", 0.4469, 0.0758, 0.3111, 0.1214]], columns=[
    "Sky selection", "Map", "theta_deg", "theta_sigma_deg", "beta_deg", "beta_sigma_deg"
])
per_map.to_csv(TABLES / "table_anchored_per_map.csv", index=False)
per_map.to_latex(
    TABLES / "table_anchored_per_map.tex", index=False,
    float_format="%.4f",
    caption="Per-map total rotations and conditional anchored birefringence estimates.",
    label="tab:permap"
)
per_map


In [ ]:

# Figure 6: per-map anchored estimates
fig, ax = plt.subplots(figsize=(6.4, 4.5))
offsets = {"Nearly full sky": -0.10, "30% Galactic mask": 0.10}
markers = {"Nearly full sky": "o", "30% Galactic mask": "s"}

for selection, group in per_map.groupby("Sky selection", sort=False):
    positions = np.arange(len(group)) + offsets[selection]
    ax.errorbar(
        group["beta_deg"], positions, xerr=group["beta_sigma_deg"],
        fmt=markers[selection], capsize=3, label=selection
    )

ax.axvline(0, lw=0.8)
ax.set_yticks(np.arange(4))
ax.set_yticklabels(["100A", "143A", "100B", "143B"])
ax.set_xlabel(r"Conditional per-map birefringence estimate $\beta_m$ [deg]")
ax.set_ylabel("Planck HFI map")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "figure_anchored_beta_per_map")
plt.show()


## 4. Multipole-range robustness

In [ ]:

robustness = pd.DataFrame([[51, 0.3616, 0.1191], [131, 0.3595, 0.1192], [211, 0.3596, 0.1193]], columns=["ell_min", "beta_deg", "sigma_deg"])
robustness.to_csv(TABLES / "table_lmin_robustness.csv", index=False)
robustness.to_latex(
    TABLES / "table_lmin_robustness.tex", index=False,
    float_format="%.4f",
    caption="Dependence of the anchored result on the minimum multipole.",
    label="tab:lmin"
)
robustness


In [ ]:

# Figure 7: minimum-multipole robustness
fig, ax = plt.subplots(figsize=(5.4, 3.6))
ax.errorbar(
    robustness["ell_min"], robustness["beta_deg"],
    yerr=robustness["sigma_deg"], fmt="o-", capsize=3,
    label="Anchored estimate"
)
ax.axhline(anchored.loc[1, "beta_deg"], ls="--", lw=0.9,
           label="Fiducial nearly full-sky result")
ax.set_xlabel(r"Minimum multipole $\ell_{\min}$")
ax.set_ylabel(r"Conditional birefringence estimate $\beta$ [deg]")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "figure_lmin_robustness")
plt.show()



## 5. Compact manuscript summary table

The closure statistic below is copied from the verified mask-0 analytic joint-covariance calculation. It should be regenerated from common simulations before using it as a precision goodness-of-fit statement.


In [ ]:

summary = pd.DataFrame([
    ["Pair-estimator Pearson correlation", correlation, np.nan, "28 map pairs"],
    ["Network closure chi-square", 8.18, 7, "Analytic shared-data covariance"],
    ["Network closure probability to exceed", stats.chi2.sf(8.18, 7), np.nan, "Analytic shared-data covariance"],
    ["Largest absolute shared-data pull", pairs["pull_shared_data"].abs().max(), np.nan, "217A-217B"],
    ["Anchored beta, nearly full sky [deg]", 0.3657, 0.1153, "Conditional on MK common-mode anchor"],
    ["Anchored beta, 30% Galactic mask [deg]", 0.4064, 0.1145, "Conditional on the same anchor"],
], columns=["Quantity", "Value", "Uncertainty or dof", "Qualification"])

summary.to_csv(TABLES / "table_results_summary.csv", index=False)
summary.to_latex(
    TABLES / "table_results_summary.tex", index=False,
    float_format="%.4f",
    caption="Summary of the principal differential-calibration and conditional anchored results.",
    label="tab:summary"
)
summary



## 6. Recommended manuscript wording generated from the tabulated results

**Differential calibration.**  
The birefringence-insensitive estimator recovers the same detector-dependent calibration pattern as the Minami–Komatsu likelihood. Across the 28 Planck HFI map pairs, the two sets of differential angles have a Pearson correlation coefficient of \(0.984\). A network-level comparison using the analytic shared-data covariance gives \(\chi^2=8.18\) for seven differential degrees of freedom. The pair-level values and the network solution should be interpreted as validation of the relative calibration pattern; neither estimator fixes the common absolute angle mode.

**Anchored application.**  
After adopting the common calibration mode inferred by the Minami–Komatsu likelihood, the total-rotation analysis yields \(\beta=0.3657\pm0.1153\) deg for the nearly full-sky selection and \(\beta=0.4064\pm0.1145\) deg for the 30% Galactic mask. These are conditional consistency results rather than an independent absolute measurement of cosmic birefringence.

**Required final validation before submission.**  
Regenerate the 30% mask comparison after correcting the mixed-mask notebook; estimate the joint MK–RelCal covariance with common realizations; estimate the covariance of the two sky selections before assigning a significance to their difference; and replace rounded MK map-level values with posterior-derived means and the full covariance.


In [ ]:

# Inventory of generated manuscript assets
generated = sorted(str(p) for p in OUTPUT.rglob("*") if p.is_file())
pd.DataFrame({"Generated asset": generated})
